In [10]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelBinarizer
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.utils.multiclass import unique_labels


class RBFClassifier(BaseEstimator, ClassifierMixin):
  """
  RBF Classifier with L2 weight penalty regularization.
  
  Parameters
  ----------
  n_centers : int, default=100
    Number of RBF centers to use
  reg_param : float, default=1e-3
    Regularization parameter (lambda) for weight penalty
  gamma : float or None, default=None
    Gamma parameter for RBF kernel. If None, automatically computed
    based on median distance between centers
  kmeans_params : dict or None, default=None
    Additional parameters to pass to KMeans clustering
  
  Attributes
  ----------
  centers_ : ndarray of shape (n_centers, n_features)
    Cluster centers found by KMeans
  coef_ : ndarray of shape (n_centers, n_classes)
    Weight matrix for RBF network
  gamma_ : float
    Actual gamma value used for RBF kernel
  """
  
  def __init__(self, n_centers=100, reg_param=1e-3, gamma=None, kmeans_params=None):
    self.n_centers = n_centers
    self.reg_param = reg_param
    self.gamma = gamma
    self.kmeans_params = kmeans_params

  def fit(self, X, y):
    """Fit RBF classifier to training data.
    
    Parameters
    ----------
    X : array-like of shape (n_samples, n_features)
      Training data
    y : array-like of shape (n_samples,)
      Target values
    """
    X, y = check_X_y(X, y)
    self._setup_classification(y)
    
    self._fit_centers(X)
    self._compute_gamma(X)
    H = self._compute_rbf_matrix(X)
    self._solve_weights(H, y)
    
    return self

  def predict(self, X):
    """Predict class labels for input data.
    
    Parameters
    ----------
    X : array-like of shape (n_samples, n_features)
      Input data
      
    Returns
    -------
    y_pred : ndarray of shape (n_samples,)
      Predicted class labels
    """
    check_is_fitted(self)
    X = check_array(X)
    
    H = self._compute_rbf_matrix(X)
    scores = H @ self.coef_
    
    return self._scores_to_labels(scores)

  def _setup_classification(self, y):
    """Initialize classification parameters and validate targets."""
    self.classes_ = unique_labels(y)
    self.n_classes_ = len(self.classes_)
    self._label_binarizer = LabelBinarizer().fit(y)

  def _fit_centers(self, X):
    """Find RBF centers using K-means clustering."""
    kmeans_params = self.kmeans_params or {}
    kmeans = KMeans(n_clusters=self.n_centers, **kmeans_params)
    kmeans.fit(X)
    self.centers_ = kmeans.cluster_centers_

  def _compute_gamma(self, X):
    """Compute gamma parameter if not provided."""
    if self.gamma is not None:
      self.gamma_ = self.gamma
      return

    # Compute median distance between centers or to data points if centers overlap
    pairwise_dists = cdist(self.centers_, self.centers_)
    np.fill_diagonal(pairwise_dists, np.inf)  # Ignore self-distances
    min_dists = np.min(pairwise_dists, axis=1)
    
    if np.all(min_dists == np.inf):  # All centers collapsed
      data_dists = cdist(X, self.centers_[:1])  # Distance to single center
      self.gamma_ = 1 / (2 * np.median(data_dists)**2)
    else:
      self.gamma_ = 1 / (2 * np.median(min_dists[min_dists > 0])**2)

  def _compute_rbf_matrix(self, X):
    """Compute RBF activation matrix using vectorized operations."""
    sq_dists = cdist(X, self.centers_, 'sqeuclidean')
    return np.exp(-self.gamma_ * sq_dists)

  def _solve_weights(self, H, y):
    """Solve for weights using regularized least squares."""
    y_onehot = self._label_binarizer.transform(y)
    Ht = H.T
    
    # Regularized system matrix
    A = Ht @ H + self.reg_param * np.eye(self.n_centers)
    # Right-hand side
    b = Ht @ y_onehot
    
    # Solve using Cholesky decomposition (efficient for symmetric PD systems)
    self.coef_ = np.linalg.solve(A, b)

  def _scores_to_labels(self, scores):
    """Convert decision scores to class labels."""
    if self.n_classes_ > 2:
      return self._label_binarizer.inverse_transform(scores)
    return np.where(scores.ravel() >= 0.5, self.classes_[1], self.classes_[0])

  def decision_function(self, X):
    """Compute decision scores for input data."""
    check_is_fitted(self)
    X = check_array(X)
    return self._compute_rbf_matrix(X) @ self.coef_

In [11]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate sample data
X, y = make_classification(n_samples=1000, n_classes=2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create and train classifier
clf = RBFClassifier()
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

Accuracy: 0.88


In [13]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from feature_engine.selection import DropCorrelatedFeatures

X, y = load_breast_cancer(return_X_y=True)

X = DropCorrelatedFeatures().fit_transform(X)

kf = KFold(n_splits=10, shuffle=True)

clf = RBFClassifier()

scores = cross_val_score(clf, X, y, cv=kf)
print(f"Cross-validation scores: {scores}")
print(f"Cross-validated accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

Cross-validation scores: [0.9122807  0.89473684 0.87719298 0.94736842 0.85964912 0.87719298
 0.84210526 0.87719298 0.87719298 0.94642857]
Cross-validated accuracy: 0.89 ± 0.03
